# VNU Feeder-Bus Grid-to-Link Assignment with `station_coordinates.csv`

This notebook calculates link-level **Popij** and **Sij** for the VNU feeder-bus network.

## Inputs

1. `Links_data.xlsx`  
   Link table with `from_station_id`, `to_station_id`, `link_name`, `link_type`, travel time, and distance fields.

2. `station_coordinates.csv`  
   Master station coordinate file. This notebook uses this file as the **only trusted coordinate source** for link endpoints.

3. `VNU_25m_Grid_Data.csv`  
   Grid centroid file. You may replace it with 50 m or 100 m centroid files and change `CELL_SIZE_M` accordingly.

## Link-type logic

| Link type | Assignment rule |
|---|---|
| `Forward` | Normal grid assignment using logit / nearest-link hard assignment |
| `Backward` | Replicates one nearest Forward link's assignment by midpoint distance |
| `U-turn` | Zero grid assignment; retained only for routing and path feasibility |

## Catchment logic

A grid can be assigned only if it has at least one candidate Forward link within the selected catchment rule. Default catchment = **0.95 km**.

The default setting is:

```python
CANDIDATE_RULE = "either"
```

Available options:

- `line`: candidate if shortest grid-to-link distance \(S_{g,ij}\leq1.5\,km\)
- `midpoint`: candidate if grid-to-link-midpoint distance \(\leq1.5\,km\)
- `either`: candidate if either condition is satisfied


# VNU Popij/Sij Calculation with Link-Type Logic and 1.5 km Catchment

This Colab notebook calculates `Popij` and `Sij` for each link using the revised link-type assignment rules and the **new grid centroid format** exported from Google Earth Engine.

Expected grid file columns include:

- `grid_id`
- `grid_size_m`
- `grid_area_km2`
- `longitude_epsg4326`, `latitude_epsg4326`
- `x_epsg32648`, `y_epsg32648`
- `grid_population_density`
- `grid_population`

Main update: the notebook no longer recalculates grid population from one uniform density. Instead, it uses `grid_population` directly and calculates `Popgij = grid_population × Fgij`, then aggregates `Popij = Σ Popgij` for each assigned Forward link.

Link-type logic remains:

- **U-turn**: zero grid assignment; routing only.
- **Forward**: normal hard assignment from eligible grid centroids.
- **Backward**: copies the assignment of the nearest Forward link by link-midpoint distance.


## Methodological basis

The procedure follows feeder-bus gridding logic: each grid is treated as a traffic zone, passengers inside one grid are assigned to one link, and grid-to-link distance is measured from the grid centroid to the corresponding link. The resulting assigned demand and average matching distance are aggregated as `Popij` and `Sij`.

For the updated grid format, each grid already has its own `grid_population`, so the model directly uses this value instead of assuming every full grid cell has identical population. This is more suitable for clipped polygon boundary cells and for low-density/household-adjusted cells.

The `Backward` and `U-turn` handling is a case-study adjustment to separate spatial demand assignment from routing feasibility.


In [ ]:
!pip -q install pyproj openpyxl

## Complete executable script

Upload `Links_data.xlsx` and `grid data` to Colab first. Then run the script below. Outputs will be saved into the `outputs/` folder.


In [3]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from pyproj import Transformer

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

# =========================
# USER PARAMETERS
# =========================
LINK_FILE = "Links_data.xlsx"
GRID_FILE = "VNU_100m_Grid_Data.csv"

# The new grid format should contain grid_population.
# Keep True to prevent accidental fallback to uniform-density calculation.
REQUIRE_GRID_POPULATION_FIELD = True

# Used only if REQUIRE_GRID_POPULATION_FIELD = False and grid_population is missing.
DEFAULT_CELL_SIZE_M = 25
DEFAULT_DENSITY_PEOPLE_PER_KM2 = 4554.6

# Maximum eligible distance from grid to candidate Forward link.
CATCHMENT_RADIUS_KM = 0.95

# Candidate rule options:
# "line"     = Sgij, shortest grid-centroid-to-link distance <= radius
# "midpoint" = grid centroid to link midpoint distance <= radius
# "either"   = line OR midpoint distance is within radius
CANDIDATE_RULE = "line"

# Chunk size controls memory usage.
CHUNK_SIZE = 2000

OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(exist_ok=True)

# =========================
# HELPER FUNCTIONS
# =========================
def resolve_path(file_name):
    candidates = [Path(file_name), Path('/content') / file_name, Path('/mnt/data') / file_name]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {file_name}. Upload it or update LINK_FILE/GRID_FILE.")


def first_existing(columns, candidates):
    normalized = {str(c).strip(): c for c in columns}
    for c in candidates:
        if c in normalized:
            return normalized[c]
    return None


def to_numeric_series(df, col, field_name):
    values = pd.to_numeric(df[col], errors='coerce')
    if values.isna().any():
        bad_count = int(values.isna().sum())
        raise ValueError(f"Column {col!r} contains {bad_count} non-numeric values and cannot be used as {field_name}.")
    return values.astype(float)


def point_segment_dist_matrix(px, py, ax, ay, bx, by):
    # Distance from n points to m line segments. Returns n x m in meters.
    px = px[:, None]
    py = py[:, None]
    ax = ax[None, :]
    ay = ay[None, :]
    bx = bx[None, :]
    by = by[None, :]

    abx = bx - ax
    aby = by - ay
    apx = px - ax
    apy = py - ay

    denom = abx * abx + aby * aby
    denom = np.where(denom == 0, 1e-12, denom)

    t = (apx * abx + apy * aby) / denom
    t = np.clip(t, 0.0, 1.0)

    cx = ax + t * abx
    cy = ay + t * aby
    return np.sqrt((px - cx) ** 2 + (py - cy) ** 2)


def point_midpoint_dist_matrix(px, py, mx, my):
    # Distance from n points to m link midpoints. Returns n x m in meters.
    px = px[:, None]
    py = py[:, None]
    mx = mx[None, :]
    my = my[None, :]
    return np.sqrt((px - mx) ** 2 + (py - my) ** 2)

# =========================
# LOAD DATA
# =========================
LINK_PATH = resolve_path(LINK_FILE)
GRID_PATH = resolve_path(GRID_FILE)

links = pd.read_excel(LINK_PATH)
grids = pd.read_csv(GRID_PATH)
links.columns = [str(c).strip() for c in links.columns]
grids.columns = [str(c).strip() for c in grids.columns]

print("Link file:", LINK_PATH)
print("Grid file:", GRID_PATH)
print("Links shape:", links.shape)
print("Grid shape:", grids.shape)

# =========================
# VALIDATE AND STANDARDIZE LINK INPUTS
# =========================
required_link_cols = [
    'from_station_id', 'to_station_id', 'link_type',
    'from_lat', 'from_lon', 'to_lat', 'to_lon'
]
missing = [c for c in required_link_cols if c not in links.columns]
if missing:
    raise ValueError(f"Missing required link columns: {missing}")

if 'link_name' not in links.columns:
    links['link_name'] = links['from_station_id'].astype(int).astype(str) + '_' + links['to_station_id'].astype(int).astype(str)

links['link_type'] = links['link_type'].astype(str).str.strip()
valid_types = {'Forward', 'Backward', 'U-turn'}
bad_types = sorted(set(links['link_type'].unique()) - valid_types)
if bad_types:
    raise ValueError(f"Unexpected link_type values: {bad_types}. Expected only {valid_types}")

print("link_type counts:")
print(links['link_type'].value_counts(dropna=False))

# =========================
# VALIDATE AND STANDARDIZE GRID INPUTS
# =========================
# New-format preferred columns are placed first.
id_candidates = ['grid_id', 'Grid_ID', 'id']
lon_candidates = ['longitude_epsg4326', 'lon', 'longitude', 'centroid_lon']
lat_candidates = ['latitude_epsg4326', 'lat', 'latitude', 'centroid_lat']
x_candidates = ['x_epsg32648', 'x_utm', 'centroid_x_utm', 'x']
y_candidates = ['y_epsg32648', 'y_utm', 'centroid_y_utm', 'y']
pop_candidates = ['grid_population', 'pop_g', 'population', 'Popg', 'pop']
density_candidates = ['grid_population_density', 'population_density', 'density_people_per_km2']
area_candidates = ['grid_area_km2', 'area_km2', 'cell_area_km2']
size_candidates = ['grid_size_m', 'cell_size_m']

grid_id_col = first_existing(grids.columns, id_candidates)
lon_col = first_existing(grids.columns, lon_candidates)
lat_col = first_existing(grids.columns, lat_candidates)
x_col = first_existing(grids.columns, x_candidates)
y_col = first_existing(grids.columns, y_candidates)
pop_col = first_existing(grids.columns, pop_candidates)
density_col = first_existing(grids.columns, density_candidates)
area_col = first_existing(grids.columns, area_candidates)
size_col = first_existing(grids.columns, size_candidates)

if grid_id_col is None:
    grids['grid_id'] = np.arange(1, len(grids) + 1)
    grid_id_col = 'grid_id'

if pop_col is None and REQUIRE_GRID_POPULATION_FIELD:
    raise ValueError(
        "The new grid file must contain a grid_population column. "
        "Set REQUIRE_GRID_POPULATION_FIELD = False only if you intentionally want the fallback density × area calculation."
    )

# =========================
# PROJECT TO UTM 48N
# =========================
transformer_to_utm = Transformer.from_crs('EPSG:4326', 'EPSG:32648', always_xy=True)
transformer_to_wgs = Transformer.from_crs('EPSG:32648', 'EPSG:4326', always_xy=True)

if x_col is None or y_col is None:
    if lon_col is None or lat_col is None:
        raise ValueError('Grid file must contain either UTM x/y columns or lon/lat columns.')
    gx, gy = transformer_to_utm.transform(
        to_numeric_series(grids, lon_col, 'grid longitude').to_numpy(),
        to_numeric_series(grids, lat_col, 'grid latitude').to_numpy()
    )
    grids['x_utm'] = gx
    grids['y_utm'] = gy
else:
    grids['x_utm'] = to_numeric_series(grids, x_col, 'grid UTM x')
    grids['y_utm'] = to_numeric_series(grids, y_col, 'grid UTM y')

if lon_col is None or lat_col is None:
    glon, glat = transformer_to_wgs.transform(grids['x_utm'].to_numpy(float), grids['y_utm'].to_numpy(float))
    grids['lon'] = glon
    grids['lat'] = glat
else:
    grids['lon'] = to_numeric_series(grids, lon_col, 'grid longitude')
    grids['lat'] = to_numeric_series(grids, lat_col, 'grid latitude')

grids['grid_id_final'] = pd.to_numeric(grids[grid_id_col], errors='coerce')
if grids['grid_id_final'].isna().any():
    raise ValueError(f"Column {grid_id_col!r} contains non-numeric grid IDs.")
grids['grid_id_final'] = grids['grid_id_final'].astype(int)

fx, fy = transformer_to_utm.transform(links['from_lon'].to_numpy(float), links['from_lat'].to_numpy(float))
tx, ty = transformer_to_utm.transform(links['to_lon'].to_numpy(float), links['to_lat'].to_numpy(float))
links['from_x_utm'] = fx
links['from_y_utm'] = fy
links['to_x_utm'] = tx
links['to_y_utm'] = ty
links['mid_x_utm'] = (links['from_x_utm'] + links['to_x_utm']) / 2
links['mid_y_utm'] = (links['from_y_utm'] + links['to_y_utm']) / 2

# =========================
# POPULATION PER GRID FROM NEW FORMAT
# =========================
if pop_col is not None:
    grids['grid_population_used'] = to_numeric_series(grids, pop_col, 'grid population')
    population_source = f"direct column: {pop_col}"
else:
    # Fallback path is intentionally explicit and should normally not be used for the new format.
    if area_col is not None:
        grids['grid_area_km2_used'] = to_numeric_series(grids, area_col, 'grid area km2')
    elif size_col is not None:
        cell_size_values = to_numeric_series(grids, size_col, 'grid size m')
        grids['grid_area_km2_used'] = (cell_size_values ** 2) / 1_000_000
    else:
        grids['grid_area_km2_used'] = (DEFAULT_CELL_SIZE_M ** 2) / 1_000_000

    if density_col is not None:
        density_values = to_numeric_series(grids, density_col, 'grid population density')
    else:
        density_values = DEFAULT_DENSITY_PEOPLE_PER_KM2

    grids['grid_population_used'] = density_values * grids['grid_area_km2_used']
    population_source = "fallback calculation: density × grid_area_km2"

if (grids['grid_population_used'] < 0).any():
    raise ValueError('grid_population contains negative values. Please check the grid file.')

# Keep these fields for traceability in outputs.
if size_col is not None:
    grids['grid_size_m_used'] = to_numeric_series(grids, size_col, 'grid size m')
else:
    grids['grid_size_m_used'] = DEFAULT_CELL_SIZE_M

if area_col is not None:
    grids['grid_area_km2_used'] = to_numeric_series(grids, area_col, 'grid area km2')
else:
    # Fallback only if grid_area_km2 is missing.
    # For full cells: area = grid_size_m × grid_size_m.
    grids['grid_area_km2_used'] = (grids['grid_size_m_used'] ** 2) / 1_000_000

if density_col is not None:
    grids['grid_population_density_used'] = to_numeric_series(grids, density_col, 'grid population density')
else:
    grids['grid_population_density_used'] = np.nan

print("Grid population source:", population_source)
print(f"Total grid population : {grids['grid_population_used'].sum():,.6f} people")
print(f"Mean grid population  : {grids['grid_population_used'].mean():,.6f} people/grid")
print(f"Min/Max grid pop.     : {grids['grid_population_used'].min():,.6f} / {grids['grid_population_used'].max():,.6f}")

# =========================
# FORWARD ASSIGNMENT
# =========================
forward = links[links['link_type'].eq('Forward')].copy().reset_index().rename(columns={'index': 'original_index'})
if len(forward) == 0:
    raise ValueError('No Forward links found. Cannot assign grids.')

A_x = forward['from_x_utm'].to_numpy(float)
A_y = forward['from_y_utm'].to_numpy(float)
B_x = forward['to_x_utm'].to_numpy(float)
B_y = forward['to_y_utm'].to_numpy(float)
M_x = forward['mid_x_utm'].to_numpy(float)
M_y = forward['mid_y_utm'].to_numpy(float)

radius_m = CATCHMENT_RADIUS_KM * 1000
assign_records = []

for start in range(0, len(grids), CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, len(grids))
    gchunk = grids.iloc[start:end]
    px = gchunk['x_utm'].to_numpy(float)
    py = gchunk['y_utm'].to_numpy(float)

    sg_m = point_segment_dist_matrix(px, py, A_x, A_y, B_x, B_y)
    mid_m = point_midpoint_dist_matrix(px, py, M_x, M_y)

    if CANDIDATE_RULE == 'line':
        eligible = sg_m <= radius_m
    elif CANDIDATE_RULE == 'midpoint':
        eligible = mid_m <= radius_m
    elif CANDIDATE_RULE == 'either':
        eligible = (sg_m <= radius_m) | (mid_m <= radius_m)
    else:
        raise ValueError("CANDIDATE_RULE must be 'line', 'midpoint', or 'either'.")

    candidate_count = eligible.sum(axis=1)
    masked_sg = np.where(eligible, sg_m, np.inf)
    best_pos = np.argmin(masked_sg, axis=1)
    best_sg_m = masked_sg[np.arange(len(gchunk)), best_pos]
    best_mid_m = mid_m[np.arange(len(gchunk)), best_pos]

    selected_prob = np.full(len(gchunk), np.nan)
    for r in range(len(gchunk)):
        if candidate_count[r] > 0:
            valid_d_km = sg_m[r, eligible[r]] / 1000.0
            d_min = best_sg_m[r] / 1000.0
            # Probability of the selected nearest link under the logit expression.
            # Assignment remains deterministic: Fg,ij = 1 for the selected link only.
            selected_prob[r] = 1.0 / np.exp(-(valid_d_km - d_min)).sum()

    has_assignment = candidate_count > 0
    for local_idx, (_, grow) in enumerate(gchunk.iterrows()):
        grid_pop = float(grow['grid_population_used'])
        if has_assignment[local_idx]:
            fp = int(best_pos[local_idx])
            frow = forward.iloc[fp]
            assign_records.append({
                'grid_id': int(grow['grid_id_final']),
                'grid_lon': float(grow['lon']),
                'grid_lat': float(grow['lat']),
                'grid_x_utm': float(grow['x_utm']),
                'grid_y_utm': float(grow['y_utm']),
                'grid_size_m': float(grow['grid_size_m_used']) if pd.notna(grow['grid_size_m_used']) else np.nan,
                'grid_area_km2': float(grow['grid_area_km2_used']) if pd.notna(grow['grid_area_km2_used']) else np.nan,
                'grid_population_density': float(grow['grid_population_density_used']) if pd.notna(grow['grid_population_density_used']) else np.nan,
                'grid_population': grid_pop,
                'Fgij': 1,
                'Popgij': grid_pop,
                'assigned': 1,
                'candidate_forward_link_count': int(candidate_count[local_idx]),
                'assigned_forward_original_index': int(frow['original_index']),
                'assigned_forward_link_name': frow['link_name'],
                'assigned_forward_from_station_id': int(frow['from_station_id']),
                'assigned_forward_to_station_id': int(frow['to_station_id']),
                'Sgij_m': float(best_sg_m[local_idx]),
                'Sgij_km': float(best_sg_m[local_idx] / 1000.0),
                'grid_to_link_midpoint_m': float(best_mid_m[local_idx]),
                'grid_to_link_midpoint_km': float(best_mid_m[local_idx] / 1000.0),
                'logit_probability_selected': float(selected_prob[local_idx]),
                'catchment_rule': CANDIDATE_RULE,
                'catchment_radius_km': CATCHMENT_RADIUS_KM,
            })
        else:
            assign_records.append({
                'grid_id': int(grow['grid_id_final']),
                'grid_lon': float(grow['lon']),
                'grid_lat': float(grow['lat']),
                'grid_x_utm': float(grow['x_utm']),
                'grid_y_utm': float(grow['y_utm']),
                'grid_size_m': float(grow['grid_size_m_used']) if pd.notna(grow['grid_size_m_used']) else np.nan,
                'grid_area_km2': float(grow['grid_area_km2_used']) if pd.notna(grow['grid_area_km2_used']) else np.nan,
                'grid_population_density': float(grow['grid_population_density_used']) if pd.notna(grow['grid_population_density_used']) else np.nan,
                'grid_population': grid_pop,
                'Fgij': 0,
                'Popgij': 0.0,
                'assigned': 0,
                'candidate_forward_link_count': 0,
                'assigned_forward_original_index': -1,
                'assigned_forward_link_name': None,
                'assigned_forward_from_station_id': np.nan,
                'assigned_forward_to_station_id': np.nan,
                'Sgij_m': np.nan,
                'Sgij_km': np.nan,
                'grid_to_link_midpoint_m': np.nan,
                'grid_to_link_midpoint_km': np.nan,
                'logit_probability_selected': np.nan,
                'catchment_rule': CANDIDATE_RULE,
                'catchment_radius_km': CATCHMENT_RADIUS_KM,
            })

assignments = pd.DataFrame(assign_records)
print("Grid assignment completed.")
print("Assigned grids  :", int(assignments['assigned'].sum()))
print("Unassigned grids:", int((assignments['assigned'] == 0).sum()))
print("Assigned share  :", assignments['assigned'].mean())
print(f"Assigned population from Popgij: {assignments['Popgij'].sum():,.6f} people")

# =========================
# AGGREGATE FORWARD POPij and Sij
# =========================
assigned_only = assignments[assignments['assigned'].eq(1)].copy()
if len(assigned_only) > 0:
    assigned_only['Sij_component_m'] = assigned_only['Sgij_m'] * assigned_only['Popgij']
    forward_agg = (
        assigned_only.groupby('assigned_forward_original_index')
        .agg(
            assigned_grid_count=('grid_id', 'count'),
            assigned_grid_area_km2=('grid_area_km2', 'sum'),
            Popij=('Popgij', 'sum'),
            assigned_grid_population_total=('grid_population', 'sum'),
            Sij_weighted_sum_m=('Sij_component_m', 'sum'),
            mean_assignment_probability=('logit_probability_selected', 'mean'),
            mean_candidate_count=('candidate_forward_link_count', 'mean'),
            min_Sgij_m=('Sgij_m', 'min'),
            max_Sgij_m=('Sgij_m', 'max'),
        )
            .reset_index()
            .rename(columns={'assigned_forward_original_index': 'original_index'})
    )
    forward_agg['Sij_m'] = np.where(
        forward_agg['Popij'] > 0,
        forward_agg['Sij_weighted_sum_m'] / forward_agg['Popij'],
        0.0
    )
    forward_agg['Sij_km'] = forward_agg['Sij_m'] / 1000.0
    forward_agg['assigned_grid_area_m2'] = forward_agg['assigned_grid_area_km2'] * 1_000_000
else:
    forward_agg = pd.DataFrame(columns=[
        'original_index', 'assigned_grid_count', 'Popij', 'assigned_grid_population_total',
        'Sij_weighted_sum_m', 'Sij_m', 'Sij_km', 'mean_assignment_probability',
        'mean_candidate_count', 'min_Sgij_m', 'max_Sgij_m'
    ])

# =========================
# APPLY LINK TYPE RULES
# =========================
result = links.copy().reset_index().rename(columns={'index': 'original_index'})
result = result.merge(forward_agg.drop(columns=['Sij_weighted_sum_m'], errors='ignore'), on='original_index', how='left')

numeric_cols = [
    'assigned_grid_count',
    'assigned_grid_area_km2',
    'assigned_grid_area_m2',
    'Popij',
    'assigned_grid_population_total',
    'Sij_m',
    'Sij_km',
    'mean_assignment_probability',
    'mean_candidate_count',
    'min_Sgij_m',
    'max_Sgij_m'
]
for c in numeric_cols:
    if c not in result.columns:
        result[c] = 0
    result[c] = result[c].fillna(0)

result['assignment_rule'] = np.select(
    [result['link_type'].eq('Forward'), result['link_type'].eq('Backward'), result['link_type'].eq('U-turn')],
    ['Direct hard assignment from eligible grid centroids using grid_population',
     'Replicate nearest Forward link assignment by midpoint distance',
     'Zero grid assignment; routing only'],
    default='Unknown'
)
result['assignment_source_link_name'] = np.where(result['link_type'].eq('Forward'), result['link_name'], '')
result['assignment_source_original_index'] = np.where(result['link_type'].eq('Forward'), result['original_index'], np.nan)
result['mapping_midpoint_distance_m'] = np.nan

# Backward replication
backward = result[result['link_type'].eq('Backward')].copy()
forward_result = result[result['link_type'].eq('Forward')].copy()
map_records = []
if len(backward) > 0 and len(forward_result) > 0:
    b_mx = backward['mid_x_utm'].to_numpy(float)[:, None]
    b_my = backward['mid_y_utm'].to_numpy(float)[:, None]
    f_mx = forward_result['mid_x_utm'].to_numpy(float)[None, :]
    f_my = forward_result['mid_y_utm'].to_numpy(float)[None, :]
    d_mid = np.sqrt((b_mx - f_mx) ** 2 + (b_my - f_my) ** 2)
    nearest_pos = np.argmin(d_mid, axis=1)

    for row_i, (_, brow) in enumerate(backward.iterrows()):
        frow = forward_result.iloc[int(nearest_pos[row_i])]
        dist_m = float(d_mid[row_i, int(nearest_pos[row_i])])
        b_idx = result['original_index'].eq(brow['original_index'])

        for col in numeric_cols:
            result.loc[b_idx, col] = frow[col]
        result.loc[b_idx, 'assignment_source_link_name'] = frow['link_name']
        result.loc[b_idx, 'assignment_source_original_index'] = frow['original_index']
        result.loc[b_idx, 'mapping_midpoint_distance_m'] = dist_m

        map_records.append({
            'backward_original_index': int(brow['original_index']),
            'backward_link_name': brow['link_name'],
            'backward_from_station_id': int(brow['from_station_id']),
            'backward_to_station_id': int(brow['to_station_id']),
            'nearest_forward_original_index': int(frow['original_index']),
            'nearest_forward_link_name': frow['link_name'],
            'nearest_forward_from_station_id': int(frow['from_station_id']),
            'nearest_forward_to_station_id': int(frow['to_station_id']),
            'midpoint_distance_m': dist_m,
            'copied_assigned_grid_count': int(frow['assigned_grid_count']),
            'copied_Popij': float(frow['Popij']),
            'copied_Sij_m': float(frow['Sij_m']),
            'copied_Sij_km': float(frow['Sij_km']),
        })
backward_mapping = pd.DataFrame(map_records)

# U-turn force zero assignment
uturn_mask = result['link_type'].eq('U-turn')
for c in numeric_cols:
    result.loc[uturn_mask, c] = 0
result.loc[uturn_mask, 'assignment_source_link_name'] = ''
result.loc[uturn_mask, 'assignment_source_original_index'] = np.nan
result.loc[uturn_mask, 'mapping_midpoint_distance_m'] = np.nan

result['unique_population_counted'] = result['link_type'].eq('Forward')
result['final_link_population_note'] = np.select(
    [result['link_type'].eq('Forward'), result['link_type'].eq('Backward'), result['link_type'].eq('U-turn')],
    ['Unique population assignment based on direct grid_population',
     'Replicated from nearest Forward link; do not add to unique coverage total',
     'No population assignment'],
    default='Unknown'
)

# =========================
# VALIDATION SUMMARY
# =========================
unique_forward_assigned_population = result.loc[result['link_type'].eq('Forward'), 'Popij'].sum()
all_grid_population = grids['grid_population_used'].sum()
all_grid_area_km2 = grids['grid_area_km2_used'].sum()
unique_forward_assigned_area_km2 = result.loc[result['link_type'].eq('Forward'), 'assigned_grid_area_km2'].sum()
assigned_grid_count = int(assignments['assigned'].sum())
unassigned_grid_count = int((assignments['assigned'] == 0).sum())
summary = pd.DataFrame([
    {'metric': 'link_file', 'value': LINK_PATH.name},
    {'metric': 'grid_file', 'value': GRID_PATH.name},
    {'metric': 'grid_population_source', 'value': population_source},
    {'metric': 'link_rows', 'value': len(links)},
    {'metric': 'grid_rows', 'value': len(grids)},
    {'metric': 'catchment_radius_km', 'value': CATCHMENT_RADIUS_KM},
    {'metric': 'candidate_rule', 'value': CANDIDATE_RULE},
    {'metric': 'forward_links', 'value': int((links['link_type'] == 'Forward').sum())},
    {'metric': 'backward_links', 'value': int((links['link_type'] == 'Backward').sum())},
    {'metric': 'uturn_links', 'value': int((links['link_type'] == 'U-turn').sum())},
    {'metric': 'assigned_grid_count', 'value': assigned_grid_count},
    {'metric': 'unassigned_grid_count', 'value': unassigned_grid_count},
    {'metric': 'assigned_grid_share', 'value': assigned_grid_count / len(grids)},
    {'metric': 'total_grid_population_from_input', 'value': all_grid_population},
    {'metric': 'unique_forward_assigned_population', 'value': unique_forward_assigned_population},
    {'metric': 'unassigned_population', 'value': all_grid_population - unique_forward_assigned_population},
    {'metric': 'assigned_population_share', 'value': unique_forward_assigned_population / all_grid_population if all_grid_population else np.nan},
    {'metric': 'final_link_level_population_sum_with_backward_replication', 'value': result['Popij'].sum()},
    {'metric': 'total_grid_area_km2_from_input', 'value': all_grid_area_km2},
{'metric': 'unique_forward_assigned_area_km2', 'value': unique_forward_assigned_area_km2},
{'metric': 'unassigned_grid_area_km2', 'value': all_grid_area_km2 - unique_forward_assigned_area_km2},
{'metric': 'assigned_area_share', 'value': unique_forward_assigned_area_km2 / all_grid_area_km2 if all_grid_area_km2 else np.nan},
{'metric': 'final_link_level_area_sum_with_backward_replication', 'value': result['assigned_grid_area_km2'].sum()},
])

# =========================
# EXPORT OUTPUTS
# =========================
preferred_cols = [
    'original_index', 'link_name', 'link_type',
    'from_station_id', 'to_station_id',
    'from_lat', 'from_lon', 'to_lat', 'to_lon',
    'from_station_name', 'to_station_name',
    'gis_route_distance_m', 'gis_route_distance_km',
    'average_travel_time_s', 'average_travel_time_min', 'average_speed_kmh',
    'assigned_grid_count',
'assigned_grid_area_km2',
'assigned_grid_area_m2',
'Popij',
'assigned_grid_population_total',
'Sij_m',
'Sij_km',
    'assignment_rule', 'assignment_source_link_name', 'assignment_source_original_index',
    'mapping_midpoint_distance_m',
    'unique_population_counted', 'final_link_population_note',
    'mean_assignment_probability', 'mean_candidate_count', 'min_Sgij_m', 'max_Sgij_m'
]
existing_preferred = [c for c in preferred_cols if c in result.columns]
remaining = [c for c in result.columns if c not in existing_preferred]
result_export = result[existing_preferred + remaining].copy()

main_csv = OUT_DIR / 'vnu_link_popij_sij_linktype_gridpop_catchment_950m.csv'
grid_csv = OUT_DIR / 'vnu_grid_assignment_forward_gridpop_catchment_950m.csv'
map_csv = OUT_DIR / 'vnu_backward_to_forward_mapping_gridpop_catchment_950m.csv'
summary_csv = OUT_DIR / 'vnu_popij_sij_assignment_summary_gridpop_950m.csv'
excel_file = OUT_DIR / 'vnu_popij_sij_linktype_gridpop_catchment_950m.xlsx'

result_export.to_csv(main_csv, index=False)
assignments.to_csv(grid_csv, index=False)
backward_mapping.to_csv(map_csv, index=False)
summary.to_csv(summary_csv, index=False)

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    result_export.to_excel(writer, sheet_name='link_popij_sij', index=False)
    assignments.to_excel(writer, sheet_name='grid_assignments', index=False)
    backward_mapping.to_excel(writer, sheet_name='backward_mapping', index=False)
    summary.to_excel(writer, sheet_name='summary', index=False)

print("\n===== VALIDATION SUMMARY =====")
print(summary.to_string(index=False))
print("\n===== SAVED OUTPUTS =====")
print(main_csv)
print(grid_csv)
print(map_csv)
print(summary_csv)
print(excel_file)


Link file: Links_data.xlsx
Grid file: VNU_100m_Grid_Data.csv
Links shape: (271, 15)
Grid shape: (1380, 11)
link_type counts:
link_type
Forward     163
Backward     66
U-turn       42
Name: count, dtype: int64
Grid population source: direct column: grid_population
Total grid population : 59,705.112750 people
Mean grid population  : 43.264574 people/grid
Min/Max grid pop.     : 0.000210 / 191.632732
Grid assignment completed.
Assigned grids  : 1380
Unassigned grids: 0
Assigned share  : 1.0
Assigned population from Popgij: 59,705.112750 people

===== VALIDATION SUMMARY =====
                                                   metric                          value
                                                link_file                Links_data.xlsx
                                                grid_file         VNU_100m_Grid_Data.csv
                                   grid_population_source direct column: grid_population
                                                link_rows        